# 06. Sigma residual and final closure

This notebook keeps the stationary alignment closure fixed and compares residual models for $R_\sigma$.  The selected interpretable closure is the affine residual $R_\sigma=a_0(r)+a_1(r)\sigma$.


# Ellipsoid sigma-residual closure, v15

The v14 notebook showed that the stationary von-Mises alignment closure is the best current angular model. It substantially improves the aligned-strain source

$$
2A\cos\alpha,
$$

where

$$
A=|S|,\qquad \alpha=2(\theta_S-\theta_g).
$$

The remaining discrepancy in $\langle\sigma\mid r\rangle$ is now most likely caused by the residual shape term

$$
R_\sigma=\dot\sigma-2A\cos\alpha.
$$

This notebook keeps the VM alignment closure fixed and compares several residual closures:

1. **VM0:** the current nonlinear damping baseline

$$
R_\sigma=c_\sigma(r)-\frac{1}{2}\kappa(r)\sinh(2\sigma).
$$

2. **VM1:** affine residual in $\sigma$

$$
R_\sigma=a_0(r)+a_1(r)\sigma.
$$

3. **VM2:** source-aware residual

$$
R_\sigma=a_0(r)+a_1(r)\sigma+a_2(r)A+a_3(r)q,
\qquad q=A\cos\alpha.
$$

4. **VM3:** direct total drift benchmark

$$
\dot\sigma=b_0(r)+b_1(r)\sigma.
$$

VM3 is less mechanistic, because it does not explicitly separate source and residual. It is included as an upper benchmark for whether a simple direct drift can reproduce the aspect-ratio statistics.

All markdown uses `$...$` and `$$...$$` only.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import i0, i1

OUT_DIR = "figs_sigma_residual_v15"
os.makedirs(OUT_DIR, exist_ok=True)

DATA_PATH = "empirical_train_mavg_out_v10b_enriched.npz"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "/mnt/data/empirical_train_mavg_out_v10b_enriched.npz"

rng = np.random.default_rng(1500)
np.set_printoptions(precision=4, suppress=True)

print("Loading", DATA_PATH)
z = np.load(DATA_PATH)
print("Keys:", z.files)

## 1. Construct intrinsic variables

The reduced variables are

$$
x=(v,\sigma,A,\omega,\alpha),\qquad r=e^v.
$$

The alignment variable is periodic:

$$
\alpha=2(\theta_S-\theta_g)\mod 2\pi.
$$

The aligned source is

$$
q=A\cos\alpha,
\qquad 2q=2A\cos\alpha.
$$

In [ ]:
def wrap_pi(x):
    return (x + np.pi) % (2*np.pi) - np.pi

def bin_index_from_r(rval, edges):
    return np.searchsorted(edges, rval, side="right") - 1

times = z["times"]
dt = float(np.median(np.diff(times)))
r_edges = z["r_edges"]
r_centers = z["r_centers"]
n_bins = len(r_centers)

s_plus = z["s_plus"]
s_cross = z["s_cross"]
omega = z["omega"]
v = z["v"]
sigma = z["sigma"]
theta_g = z["theta_g"]
theta_s = z["theta_s"]
r = z["r"]

A = np.sqrt(s_plus**2 + s_cross**2)
alpha = wrap_pi(2*(theta_s - theta_g))
q = A*np.cos(alpha)

n_seed, n_time = sigma.shape
print("n_seed, n_time:", n_seed, n_time)
print("dt:", dt)
print("r bins:", r_centers)
print("sigma range:", float(np.nanmin(sigma)), float(np.nanmax(sigma)))

## 2. Short-time increments

The empirical generator is estimated from finite lags. We use the same convention as v14: a short lag reduces raw derivative noise but remains small compared to the full cloud-evolution time.

For the angular variable we use the wrapped increment

$$
\Delta\alpha=\operatorname{atan2}(\sin(\alpha_{t+\Delta}-\alpha_t),\cos(\alpha_{t+\Delta}-\alpha_t)).
$$

In [ ]:
lag_steps = 5
lag_dt = lag_steps*dt
sl0 = np.s_[:, :-lag_steps]
sl1 = np.s_[:, lag_steps:]

v0, v1 = v[sl0], v[sl1]
s0, s1 = sigma[sl0], sigma[sl1]
A0, A1 = A[sl0], A[sl1]
w0, w1 = omega[sl0], omega[sl1]
a0, a1 = alpha[sl0], alpha[sl1]
r0 = r[sl0]

bv_emp = (v1-v0)/lag_dt
bA_emp = (A1-A0)/lag_dt
bw_emp = (w1-w0)/lag_dt
bs_emp = (s1-s0)/lag_dt
ba_emp = wrap_pi(a1-a0)/lag_dt

source0 = 2*A0*np.cos(a0)
q0 = A0*np.cos(a0)
Rsig_emp = bs_emp - source0

rf = r0.ravel()
vf = v0.ravel()
sf = s0.ravel()
Af = A0.ravel()
wf = w0.ravel()
af = a0.ravel()
qf = q0.ravel()

bvf = bv_emp.ravel()
bAf = bA_emp.ravel()
bwf = bw_emp.ravel()
bsf = bs_emp.ravel()
baf = ba_emp.ravel()
sourcef = source0.ravel()
Rsf = Rsig_emp.ravel()

binf = bin_index_from_r(rf, r_edges)
valid = np.isfinite(rf+sf+Af+wf+af+baf+Rsf+sourcef) & (binf >= 0) & (binf < n_bins)
print("valid increment samples:", int(valid.sum()))
print("lag_dt:", lag_dt)

## 3. Utility routines

The notebook uses radial binning, ridge fits, and interpolation in $r$. This is intentionally conservative. The goal is to compare residual model structures, not to optimize a black-box learner.

In [ ]:
def bin_mean(x, b, n_bins, min_count=10):
    out = np.full(n_bins, np.nan)
    cnt = np.zeros(n_bins, dtype=int)
    for k in range(n_bins):
        sel = (b == k) & np.isfinite(x)
        cnt[k] = int(sel.sum())
        if cnt[k] >= min_count:
            out[k] = np.nanmean(x[sel])
    return out, cnt

def bin_var(x, b, n_bins, min_count=10):
    out = np.full(n_bins, np.nan)
    for k in range(n_bins):
        sel = (b == k) & np.isfinite(x)
        if sel.sum() >= min_count:
            out[k] = np.nanvar(x[sel])
    return out

def fill_nan(arr):
    arr = np.asarray(arr, float).copy()
    idx = np.arange(len(arr))
    good = np.isfinite(arr)
    if good.sum() == 0:
        return np.zeros_like(arr)
    if good.sum() == 1:
        arr[~good] = arr[good][0]
        return arr
    arr[~good] = np.interp(idx[~good], idx[good], arr[good])
    return arr

def interp_by_r(rval, arr):
    arr = fill_nan(arr)
    return np.interp(np.asarray(rval), r_centers, arr, left=arr[0], right=arr[-1])

def ridge_fit(X, y, lam=1e-6):
    X = np.asarray(X, float)
    y = np.asarray(y, float)
    ok = np.all(np.isfinite(X), axis=1) & np.isfinite(y)
    X = X[ok]
    y = y[ok]
    if len(y) < X.shape[1] + 5:
        return np.zeros(X.shape[1]), np.nan, len(y)
    XtX = X.T @ X
    Xty = X.T @ y
    reg = lam*np.eye(X.shape[1])
    reg[0, 0] = 0.0
    beta = np.linalg.solve(XtX + reg, Xty)
    resid = y - X @ beta
    return beta, float(np.mean(resid**2)), len(y)

def A1_of_K(K):
    K = np.asarray(K, float)
    return i1(K) / np.maximum(i0(K), 1e-300)

def invert_A1(m, max_iter=30):
    m = np.asarray(m, float)
    m_clip = np.clip(m, 0.0, 0.985)
    K = np.empty_like(m_clip)
    mask1 = m_clip < 0.53
    mask2 = (m_clip >= 0.53) & (m_clip < 0.85)
    mask3 = m_clip >= 0.85
    K[mask1] = 2*m_clip[mask1] + m_clip[mask1]**3 + 5*m_clip[mask1]**5/6
    K[mask2] = -0.4 + 1.39*m_clip[mask2] + 0.43/(1 - m_clip[mask2])
    K[mask3] = 1/(m_clip[mask3]**3 - 4*m_clip[mask3]**2 + 3*m_clip[mask3] + 1e-12)
    for _ in range(max_iter):
        A1 = A1_of_K(K)
        der = 1 - A1/np.maximum(K, 1e-8) - A1**2
        K = K - (A1 - m_clip)/np.maximum(der, 1e-8)
        K = np.clip(K, 0.0, 100.0)
    return K

## 4. Empirical validation statistics and budgets

The empirical reference includes both state statistics and increment-based budgets:

$$
\dot\sigma = 2A\cos\alpha + R_\sigma.
$$

A successful closure should match not only $\langle\sigma\mid r\rangle$, but also the mean source, residual, and total drift by $r$.

In [ ]:
def empirical_state_by_bin():
    b = bin_index_from_r(r.ravel(), r_edges)
    sig = sigma.ravel()
    Aval = A.ravel()
    aval = alpha.ravel()
    qval = Aval*np.cos(aval)
    out = {}
    out["sigma"], out["count"] = bin_mean(sig, b, n_bins, min_count=10)
    out["A"], _ = bin_mean(Aval, b, n_bins, min_count=10)
    out["cos"], _ = bin_mean(np.cos(aval), b, n_bins, min_count=10)
    out["source"], _ = bin_mean(2*qval, b, n_bins, min_count=10)
    return out

def empirical_increment_budget():
    out = {}
    out["dot_sigma"], out["count_inc"] = bin_mean(bsf[valid], binf[valid], n_bins, min_count=10)
    out["source"], _ = bin_mean(sourcef[valid], binf[valid], n_bins, min_count=10)
    out["R_sigma"], _ = bin_mean(Rsf[valid], binf[valid], n_bins, min_count=10)
    return out

emp = empirical_state_by_bin()
emp_budget = empirical_increment_budget()
print("Empirical sigma:", emp["sigma"])
print("Empirical source, state:", emp["source"])
print("Empirical source, increments:", emp_budget["source"])
print("Empirical R_sigma:", emp_budget["R_sigma"])
print("Empirical dot_sigma:", emp_budget["dot_sigma"])
print("Counts:", emp["count"])

## 5. Common closures for $v$, $A$, $\omega$, and VM alignment

We keep the v14 closures for $v$, $A$, $\omega$, and $\alpha$.

The VM alignment closure fits

$$
\mathbb E[\cos\alpha\mid r,\sigma,A]
=m_0(r)+m_1(r)\sigma+m_2(r)A+m_3(r)\sigma A,
$$

then maps the target mean to a von-Mises concentration $K$.

In [ ]:
b_v = np.full(n_bins, np.nan); D_v = np.full(n_bins, np.nan)
A_bar = np.full(n_bins, np.nan); lambda_A = np.full(n_bins, np.nan); D_A = np.full(n_bins, np.nan)
lambda_w = np.full(n_bins, np.nan); D_w = np.full(n_bins, np.nan)
D_alpha = np.full(n_bins, np.nan)

for k in range(n_bins):
    sel = valid & (binf == k)
    if sel.sum() < 50:
        continue
    b_v[k] = np.mean(bvf[sel])
    D_v[k] = 0.5*np.var((v1-v0).ravel()[sel]) / lag_dt

    A00 = Af[sel]
    A11 = A1.ravel()[sel]
    A_bar[k] = np.mean(A00)
    x0c = A00 - A_bar[k]
    x1c = A11 - A_bar[k]
    var0 = np.var(x0c)
    if var0 > 1e-12:
        rhoA = np.mean(x0c*x1c)/var0
        rhoA = np.clip(rhoA, 1e-4, 0.999)
        lambda_A[k] = -np.log(rhoA)/lag_dt
        D_A[k] = lambda_A[k]*var0
    else:
        lambda_A[k] = 1.0; D_A[k] = 1e-8

    w00 = wf[sel]
    w11 = w1.ravel()[sel]
    varw = np.var(w00)
    if varw > 1e-12:
        rhow = np.mean(w00*w11)/varw
        rhow = np.clip(rhow, 1e-4, 0.999)
        lambda_w[k] = -np.log(rhow)/lag_dt
        D_w[k] = lambda_w[k]*varw
    else:
        lambda_w[k] = 1.0; D_w[k] = 1e-8

    D_alpha[k] = 0.5*np.var(wrap_pi(a1-a0).ravel()[sel]) / lag_dt

# VM alignment fit
vm_beta = np.zeros((n_bins, 4))
vm_mse = np.full(n_bins, np.nan)
for k in range(n_bins):
    sel = valid & (binf == k)
    if sel.sum() < 100:
        continue
    X = np.column_stack([np.ones(sel.sum()), sf[sel], Af[sel], sf[sel]*Af[sel]])
    y = np.cos(af[sel])
    beta, mse, n = ridge_fit(X, y, lam=1e-3)
    vm_beta[k] = beta
    vm_mse[k] = mse

print("b_v:", b_v)
print("A_bar:", A_bar)
print("lambda_A:", lambda_A)
print("lambda_w:", lambda_w)
print("D_alpha:", D_alpha)
print("VM beta [1, sigma, A, sigma*A]:")
print(vm_beta)
print("VM MSE:", vm_mse)

def mcos_vm(rval, sig, Aval):
    b0 = interp_by_r(rval, vm_beta[:,0])
    b1 = interp_by_r(rval, vm_beta[:,1])
    b2 = interp_by_r(rval, vm_beta[:,2])
    b3 = interp_by_r(rval, vm_beta[:,3])
    return np.clip(b0 + b1*sig + b2*Aval + b3*sig*Aval, -0.95, 0.95)

def K_vm(rval, sig, Aval):
    return invert_A1(np.maximum(mcos_vm(rval, sig, Aval), 0.0))

## 6. Fit residual closures

We fit four residual or drift closures per radial bin.

For VM0, VM1, and VM2 we fit $R_\sigma$ directly. For VM3 we fit the total drift $\dot\sigma$ directly.

The diffusion $D_\sigma$ is estimated from the residual increment variance for each closure.

In [ ]:
# Parameter arrays and diffusion arrays.
par_VM0 = np.full((n_bins, 2), np.nan)   # [c, kappa]
par_VM1 = np.full((n_bins, 2), np.nan)   # [a0, a1]
par_VM2 = np.full((n_bins, 4), np.nan)   # [a0, a1, a2, a3]
par_VM3 = np.full((n_bins, 2), np.nan)   # [b0, b1] for total sigma drift

Dsig_VM0 = np.full(n_bins, np.nan)
Dsig_VM1 = np.full(n_bins, np.nan)
Dsig_VM2 = np.full(n_bins, np.nan)
Dsig_VM3 = np.full(n_bins, np.nan)

mse_VM0 = np.full(n_bins, np.nan)
mse_VM1 = np.full(n_bins, np.nan)
mse_VM2 = np.full(n_bins, np.nan)
mse_VM3 = np.full(n_bins, np.nan)

for k in range(n_bins):
    sel = valid & (binf == k)
    if sel.sum() < 100:
        continue

    # VM0: R = c - 0.5*kappa*sinh(2sigma)
    X0 = np.column_stack([np.ones(sel.sum()), np.sinh(2*np.clip(sf[sel], 0, 5))])
    beta0, mse0, n = ridge_fit(X0, Rsf[sel], lam=1e-4)
    c0 = beta0[0]
    kap0 = max(0.0, -2*beta0[1])
    pred0 = c0 - 0.5*kap0*np.sinh(2*np.clip(sf[sel], 0, 5))
    par_VM0[k] = [c0, kap0]
    mse_VM0[k] = np.mean((Rsf[sel]-pred0)**2)
    Dsig_VM0[k] = 0.5*np.var((Rsf[sel]-pred0)*lag_dt)/lag_dt

    # VM1: R = a0 + a1*sigma
    X1 = np.column_stack([np.ones(sel.sum()), sf[sel]])
    beta1, mse1, n = ridge_fit(X1, Rsf[sel], lam=1e-3)
    pred1 = X1 @ beta1
    par_VM1[k] = beta1
    mse_VM1[k] = mse1
    Dsig_VM1[k] = 0.5*np.var((Rsf[sel]-pred1)*lag_dt)/lag_dt

    # VM2: R = a0 + a1*sigma + a2*A + a3*q
    X2 = np.column_stack([np.ones(sel.sum()), sf[sel], Af[sel], qf[sel]])
    beta2, mse2, n = ridge_fit(X2, Rsf[sel], lam=5e-3)
    pred2 = X2 @ beta2
    par_VM2[k] = beta2
    mse_VM2[k] = mse2
    Dsig_VM2[k] = 0.5*np.var((Rsf[sel]-pred2)*lag_dt)/lag_dt

    # VM3: total drift dot sigma = b0 + b1*sigma
    X3 = np.column_stack([np.ones(sel.sum()), sf[sel]])
    beta3, mse3, n = ridge_fit(X3, bsf[sel], lam=1e-3)
    pred3 = X3 @ beta3
    par_VM3[k] = beta3
    mse_VM3[k] = mse3
    Dsig_VM3[k] = 0.5*np.var((bsf[sel]-pred3)*lag_dt)/lag_dt

print("VM0 params [c,kappa]:")
print(par_VM0)
print("VM1 params [a0,a1]:")
print(par_VM1)
print("VM2 params [a0,a1,a2,a3]:")
print(par_VM2)
print("VM3 params [b0,b1] total drift:")
print(par_VM3)
print("Residual MSE VM0:", mse_VM0)
print("Residual MSE VM1:", mse_VM1)
print("Residual MSE VM2:", mse_VM2)
print("Total drift MSE VM3:", mse_VM3)

## 7. Residual functions and forward simulation

All models use the same $v$, $A$, $\omega$, and VM alignment closures. They differ only in the $\sigma$ equation.

For VM0, VM1, and VM2:

$$
d\sigma=\left[2A\cos\alpha+R_\sigma^{\rm model}\right]dt+\sqrt{2D_\sigma}dW_\sigma.
$$

For VM3:

$$
d\sigma=b_\sigma^{\rm model}(r,\sigma)dt+\sqrt{2D_\sigma}dW_\sigma.
$$

In [ ]:
def R_model(model, rval, sig, Aval, aval):
    qval = Aval*np.cos(aval)
    if model == "VM0":
        c = interp_by_r(rval, par_VM0[:,0])
        kap = np.maximum(interp_by_r(rval, par_VM0[:,1]), 0.0)
        return c - 0.5*kap*np.sinh(2*np.clip(sig, 0, 5))
    if model == "VM1":
        a0 = interp_by_r(rval, par_VM1[:,0])
        a1 = interp_by_r(rval, par_VM1[:,1])
        return a0 + a1*sig
    if model == "VM2":
        a0 = interp_by_r(rval, par_VM2[:,0])
        a1 = interp_by_r(rval, par_VM2[:,1])
        a2 = interp_by_r(rval, par_VM2[:,2])
        a3 = interp_by_r(rval, par_VM2[:,3])
        return a0 + a1*sig + a2*Aval + a3*qval
    raise ValueError(model)

def b_total_VM3(rval, sig):
    b0 = interp_by_r(rval, par_VM3[:,0])
    b1 = interp_by_r(rval, par_VM3[:,1])
    return b0 + b1*sig

def Dsig_for_model(model, rval):
    if model == "VM0": return np.maximum(interp_by_r(rval, Dsig_VM0), 1e-10)
    if model == "VM1": return np.maximum(interp_by_r(rval, Dsig_VM1), 1e-10)
    if model == "VM2": return np.maximum(interp_by_r(rval, Dsig_VM2), 1e-10)
    if model == "VM3": return np.maximum(interp_by_r(rval, Dsig_VM3), 1e-10)
    raise ValueError(model)

def simulate_model(model="VM0", n_sim=300, seed=1510, alpha_D_scale=1.0):
    local_rng = np.random.default_rng(seed)

    # Initialize from empirical t=0 for v, sigma, A, omega; alpha from full empirical distribution
    idx0 = local_rng.integers(0, n_seed, size=n_sim)
    vv = v[idx0, 0].copy()
    ss = sigma[idx0, 0].copy()
    AA = A[idx0, 0].copy()
    ww = omega[idx0, 0].copy()

    flat_idx = local_rng.integers(0, alpha.size, size=n_sim)
    aa = alpha.ravel()[flat_idx].copy()

    out_v = np.zeros((n_sim, n_time))
    out_sig = np.zeros_like(out_v)
    out_A = np.zeros_like(out_v)
    out_w = np.zeros_like(out_v)
    out_alpha = np.zeros_like(out_v)
    out_source = np.zeros_like(out_v)
    out_R = np.zeros_like(out_v)
    out_drift_sigma = np.zeros_like(out_v)

    for t in range(n_time):
        rr = np.exp(vv)
        src = 2*AA*np.cos(aa)
        if model == "VM3":
            RR = np.nan*np.ones_like(ss)
            drift_sig = b_total_VM3(rr, ss)
        else:
            RR = R_model(model, rr, ss, AA, aa)
            drift_sig = src + RR

        out_v[:, t] = vv
        out_sig[:, t] = ss
        out_A[:, t] = AA
        out_w[:, t] = ww
        out_alpha[:, t] = aa
        out_source[:, t] = src
        out_R[:, t] = RR
        out_drift_sigma[:, t] = drift_sig

        if t == n_time - 1:
            break

        rr = np.exp(vv)
        bv = interp_by_r(rr, b_v)
        Dv = np.maximum(interp_by_r(rr, D_v), 1e-10)

        Abar = interp_by_r(rr, A_bar)
        lamA = np.maximum(interp_by_r(rr, lambda_A), 0.0)
        DA = np.maximum(interp_by_r(rr, D_A), 1e-10)

        lamw = np.maximum(interp_by_r(rr, lambda_w), 0.0)
        Dw = np.maximum(interp_by_r(rr, D_w), 1e-10)

        Daa = np.maximum(interp_by_r(rr, D_alpha), 1e-10)*alpha_D_scale
        Kaa = K_vm(rr, ss, AA)
        Ds = Dsig_for_model(model, rr)

        vv = vv + bv*dt + np.sqrt(2*Dv*dt)*local_rng.normal(size=n_sim)
        AA = AA + (-lamA*(AA - Abar))*dt + np.sqrt(2*DA*dt)*local_rng.normal(size=n_sim)
        AA = np.maximum(AA, 1e-6)
        ww = ww + (-lamw*ww)*dt + np.sqrt(2*Dw*dt)*local_rng.normal(size=n_sim)

        aa = aa + (-Daa*Kaa*np.sin(aa))*dt + np.sqrt(2*Daa*dt)*local_rng.normal(size=n_sim)
        aa = wrap_pi(aa)

        src = 2*AA*np.cos(aa)
        if model == "VM3":
            drift_sig = b_total_VM3(np.exp(vv), ss)
        else:
            drift_sig = src + R_model(model, np.exp(vv), ss, AA, aa)

        ss = ss + drift_sig*dt + np.sqrt(2*Ds*dt)*local_rng.normal(size=n_sim)
        ss = np.clip(ss, 0.0, 5.0)

    return {
        "model": model,
        "v": out_v,
        "r": np.exp(out_v),
        "sigma": out_sig,
        "A": out_A,
        "omega": out_w,
        "alpha": out_alpha,
        "source": out_source,
        "R": out_R,
        "drift_sigma": out_drift_sigma,
    }

## 8. Simulate the residual-closure hierarchy

The same random seed family is used for all models. The comparison is therefore mainly structural: which residual closure gives the right stationary aspect-ratio statistics and the right source-sink budget?

In [ ]:
models = ["VM0", "VM1", "VM2", "VM3"]
sims = {}
for i, m in enumerate(models):
    print("Simulating", m)
    sims[m] = simulate_model(model=m, n_sim=360, seed=1510+i)
print("done")

## 9. Diagnostics by $r$

We compare

$$
\langle\sigma\mid r\rangle,
\qquad
\langle 2A\cos\alpha\mid r\rangle,
\qquad
\langle R_\sigma\mid r\rangle,
\qquad
\langle\dot\sigma\mid r\rangle.
$$

For VM3, $R_\sigma$ is not defined because the model fits the total drift directly.

In [ ]:
def diagnostics_from_sim(sim):
    b = bin_index_from_r(sim["r"].ravel(), r_edges)
    out = {}
    out["sigma"], out["count"] = bin_mean(sim["sigma"].ravel(), b, n_bins, min_count=10)
    out["A"], _ = bin_mean(sim["A"].ravel(), b, n_bins, min_count=10)
    out["cos"], _ = bin_mean(np.cos(sim["alpha"].ravel()), b, n_bins, min_count=10)
    out["source"], _ = bin_mean(sim["source"].ravel(), b, n_bins, min_count=10)
    out["R"], _ = bin_mean(sim["R"].ravel(), b, n_bins, min_count=10)
    out["drift_sigma"], _ = bin_mean(sim["drift_sigma"].ravel(), b, n_bins, min_count=10)
    return out

diag = {m: diagnostics_from_sim(sims[m]) for m in models}

print("Empirical sigma:", emp["sigma"])
for m in models:
    print(m, "sigma:", diag[m]["sigma"])
print("\nEmpirical source:", emp_budget["source"])
for m in models:
    print(m, "source:", diag[m]["source"])
print("\nEmpirical R_sigma:", emp_budget["R_sigma"])
for m in models:
    print(m, "R:", diag[m]["R"])
print("\nEmpirical dot_sigma:", emp_budget["dot_sigma"])
for m in models:
    print(m, "dot sigma drift:", diag[m]["drift_sigma"])

## 10. Plots

The most important plot is the comparison of $\langle\sigma\mid r\rangle$. The budget plots then explain why a model succeeds or fails.

In [ ]:
def plot_models(key, ylabel, fname, include_emp=True, emp_arr=None):
    plt.figure(figsize=(7.0, 4.6))
    if include_emp:
        if emp_arr is None:
            emp_arr = emp[key]
        plt.plot(r_centers, emp_arr, "o-", label="empirical", lw=2)
    styles = {"VM0":"s--", "VM1":"d--", "VM2":"^--", "VM3":"v-."}
    for m in models:
        plt.plot(r_centers, diag[m][key], styles[m], label=m)
    plt.xscale("log")
    plt.xlabel("r")
    plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    path = os.path.join(OUT_DIR, fname)
    plt.savefig(path, dpi=180)
    plt.show()
    print("saved", path)

plot_models("sigma", r"$\langle\sigma\mid r\rangle$", "sigma_mean_residual_closure_comparison.png")
plot_models("source", r"$\langle 2A\cos\alpha\mid r\rangle$", "source_residual_closure_comparison.png", emp_arr=emp_budget["source"])
plot_models("R", r"$\langle R_\sigma\mid r\rangle$", "R_sigma_residual_closure_comparison.png", emp_arr=emp_budget["R_sigma"])
plot_models("drift_sigma", r"$\langle\dot\sigma\mid r\rangle$", "dot_sigma_residual_closure_comparison.png", emp_arr=emp_budget["dot_sigma"])

# Source ratio remains useful for checking that VM alignment did not get broken.
eps = 1e-12
plt.figure(figsize=(7.0, 4.2))
plt.axhline(1.0, color="k", lw=1, alpha=0.5)
for m, sty in zip(models, ["s--", "d--", "^--", "v-."]):
    ratio = diag[m]["source"] / np.maximum(emp_budget["source"], eps)
    plt.plot(r_centers, ratio, sty, label=m)
plt.xscale("log")
plt.xlabel("r")
plt.ylabel(r"$\langle 2A\cos\alpha\rangle_{\rm model}/\langle 2A\cos\alpha\rangle_{\rm emp}$")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
path = os.path.join(OUT_DIR, "source_ratio_residual_closure_comparison.png")
plt.savefig(path, dpi=180)
plt.show()
print("saved", path)

## 11. Distribution checks

Means can be misleading. We therefore compare the distribution of $\sigma$ in selected $r$-bins.

In [ ]:
selected_bins = [1, 2, 3, 4, 5]
for k in selected_bins:
    plt.figure(figsize=(7.0, 4.2))
    emp_sel = (bin_index_from_r(r.ravel(), r_edges) == k)
    plt.hist(sigma.ravel()[emp_sel], bins=35, density=True, alpha=0.35, label="empirical")
    for m in models:
        bsim = bin_index_from_r(sims[m]["r"].ravel(), r_edges)
        sel = (bsim == k)
        if sel.sum() > 20:
            plt.hist(sims[m]["sigma"].ravel()[sel], bins=35, density=True, histtype="step", lw=1.6, label=m)
    plt.xlabel(r"$\sigma$")
    plt.ylabel("density")
    plt.title(f"sigma distribution, r-bin {k}, r≈{r_centers[k]:.3g}")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    path = os.path.join(OUT_DIR, f"sigma_distribution_bin{k}.png")
    plt.savefig(path, dpi=180)
    plt.show()
    print("saved", path)

## 12. Compact score table

The score is the root-mean-square error of selected binned means. This is only a guide; the plots and distribution checks remain more informative.

In [ ]:
def rmse(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((a[ok]-b[ok])**2)))

score_rows = []
for m in models:
    score_rows.append({
        "model": m,
        "RMSE_sigma_mean": rmse(diag[m]["sigma"], emp["sigma"]),
        "RMSE_source": rmse(diag[m]["source"], emp_budget["source"]),
        "RMSE_R_sigma": rmse(diag[m]["R"], emp_budget["R_sigma"]),
        "RMSE_dot_sigma": rmse(diag[m]["drift_sigma"], emp_budget["dot_sigma"]),
    })

for row in score_rows:
    print(row)

## 13. Reading the outcome

Use the following logic.

If VM0 is already best, the original nonlinear damping residual is sufficient and the remaining error is probably in finite sampling or the VM alignment closure.

If VM1 improves, the residual is better described as a simple affine drift in $\sigma$ than as a pure $\sinh(2\sigma)$ damping.

If VM2 improves, the Lowner--John residual depends on the same geometric source variables that drive elongation, especially $A$ and $q=A\cos\alpha$.

If VM3 improves strongly, the most stable reduced model may be an effective one-dimensional drift for $\sigma$ rather than an explicitly separated source--sink model.

The preferred model should not only minimize RMSE of $\langle\sigma\mid r\rangle$; it should also preserve the empirical source--sink interpretation.

In [ ]:
# Save results for follow-up interpretation.
np.savez_compressed(
    "sigma_residual_closure_v15_results.npz",
    r_edges=r_edges,
    r_centers=r_centers,
    emp_sigma=emp["sigma"],
    emp_source=emp_budget["source"],
    emp_R_sigma=emp_budget["R_sigma"],
    emp_dot_sigma=emp_budget["dot_sigma"],
    par_VM0=par_VM0,
    par_VM1=par_VM1,
    par_VM2=par_VM2,
    par_VM3=par_VM3,
    Dsig_VM0=Dsig_VM0,
    Dsig_VM1=Dsig_VM1,
    Dsig_VM2=Dsig_VM2,
    Dsig_VM3=Dsig_VM3,
    VM0_sigma=diag["VM0"]["sigma"],
    VM1_sigma=diag["VM1"]["sigma"],
    VM2_sigma=diag["VM2"]["sigma"],
    VM3_sigma=diag["VM3"]["sigma"],
    VM0_source=diag["VM0"]["source"],
    VM1_source=diag["VM1"]["source"],
    VM2_source=diag["VM2"]["source"],
    VM3_source=diag["VM3"]["source"],
    VM0_R=diag["VM0"]["R"],
    VM1_R=diag["VM1"]["R"],
    VM2_R=diag["VM2"]["R"],
    VM3_R=diag["VM3"]["R"],
    VM0_dot_sigma=diag["VM0"]["drift_sigma"],
    VM1_dot_sigma=diag["VM1"]["drift_sigma"],
    VM2_dot_sigma=diag["VM2"]["drift_sigma"],
    VM3_dot_sigma=diag["VM3"]["drift_sigma"],
)
print("saved sigma_residual_closure_v15_results.npz")